# Earnings Profit → 3–5 Day Trend Model

用**财报利润数据**预测个股在财报公布后 **3–5 个交易日**的价格走势（“财报漂移” post-earnings drift）。

**思路**

1. 对每只股票，拉取历史财报事件（EPS 预期 vs 实际）和季度利润表（营收、净利润）。
2. 为每个财报事件构造**利润特征**：EPS 惊喜度、营收同比、净利润同比、净利率及其变化、财报前动量。
3. 给每个事件打上**标签**：未来 3 / 5 交易日的前向收益与涨跌方向。
4. 在汇总的、按时间排序的事件表上训练分类器（方向）与回归器（幅度），并样本外评估。

> 数据源用 `yfinance`，无需 API key，在 Colab / 本地都能直接跑。要换自己的股票，改下面的 `TICKERS` 即可。

## 1. 安装依赖

In [ ]:
%pip install -q yfinance lxml pandas numpy scikit-learn matplotlib

> **（可选）受限网络 / 代理环境**：若 yfinance 报 `Connection reset by peer`（企业/代理会重置 curl_cffi 的浏览器 TLS 指纹），运行下面这格把 yfinance 换成普通 requests 会话。Colab 上**无需**执行。

In [ ]:
# 仅在代理/受限网络下需要（Colab 可跳过）
# import os, requests
# os.environ["REQUESTS_CA_BUNDLE"] = "/path/to/ca-bundle.crt"  # 若有自签 CA
# _s = requests.Session(); _s.headers.update({"User-Agent": "Mozilla/5.0"})
# import earnings_trend as et; et.set_session(_s)

## 2. 配置（在这里改你的股票池）

In [ ]:
# 股票池 —— 换成你自己关心的代码即可
TICKERS = ["AAPL", "MSFT", "NVDA", "GOOGL", "AMZN",
           "META", "TSLA", "AMD", "NFLX", "JPM"]

# “3–5 天”的交易日地平线
HORIZONS = (3, 5)

## 3. 导入模块

核心逻辑在 `earnings_trend.py`。在 Colab 上请先把该文件上传，或 clone 仓库后把目录加入 path。

In [ ]:
import sys, os
# 让 notebook 无论在哪个工作目录都能找到同目录的 earnings_trend.py
sys.path.insert(0, os.path.dirname(os.path.abspath("earnings_trend.py")))

import importlib
import earnings_trend as et
importlib.reload(et)
et.HORIZONS = HORIZONS
print("module loaded, features:", et.FEATURE_COLS)

## 4. 拉取数据 + 构造事件表

每一行 = 一个历史财报事件，包含利润特征与前向走势。

In [ ]:
df = et.build_dataset(TICKERS)
print(f"\n共 {len(df)} 个财报事件，跨 {df['ticker'].nunique()} 只股票")
df.tail(8)

## 5. 探索性分析：利润特征 vs 3–5 天走势

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# 5a. 相关性热图：特征与前向收益
cols = et.FEATURE_COLS + [f"fwd_ret_{h}d" for h in HORIZONS]
corr = df[cols].corr()
fig, ax = plt.subplots(figsize=(9, 7))
im = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(cols))); ax.set_xticklabels(cols, rotation=45, ha="right")
ax.set_yticks(range(len(cols))); ax.set_yticklabels(cols)
for i in range(len(cols)):
    for j in range(len(cols)):
        ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", fontsize=7)
fig.colorbar(im, fraction=0.046, pad=0.04)
ax.set_title("Correlation: profit features vs 3-5d forward return")
plt.tight_layout(); plt.show()

In [ ]:
# 5b. EPS 惊喜度 vs 前向收益（核心假设：超预期→上涨漂移）
fig, axes = plt.subplots(1, len(HORIZONS), figsize=(6 * len(HORIZONS), 4.5))
if len(HORIZONS) == 1: axes = [axes]
for ax, h in zip(axes, HORIZONS):
    sub = df.dropna(subset=["eps_surprise_pct", f"fwd_ret_{h}d"])
    ax.scatter(sub["eps_surprise_pct"], sub[f"fwd_ret_{h}d"], alpha=0.5, s=25)
    if len(sub) > 2:
        m, b = np.polyfit(sub["eps_surprise_pct"], sub[f"fwd_ret_{h}d"], 1)
        xs = np.linspace(sub["eps_surprise_pct"].min(), sub["eps_surprise_pct"].max(), 50)
        ax.plot(xs, m * xs + b, "r-", lw=2, label=f"slope={m:.3f}")
        ax.legend()
    ax.axhline(0, color="gray", lw=0.7); ax.axvline(0, color="gray", lw=0.7)
    ax.set_xlabel("EPS surprise (%)"); ax.set_ylabel(f"{h}-day forward return (%)")
    ax.set_title(f"EPS surprise vs {h}d trend")
plt.tight_layout(); plt.show()

In [ ]:
# 5c. 按利润惊喜分组，看平均 3-5 天走势与胜率
def bucket(x):
    if x <= -5: return "1. 大幅不及 (<=-5%)"
    if x < 0:   return "2. 小幅不及 (-5~0%)"
    if x < 5:   return "3. 小幅超预期 (0~5%)"
    return "4. 大幅超预期 (>=5%)"

g = df.dropna(subset=["eps_surprise_pct"]).copy()
g["surprise_bucket"] = g["eps_surprise_pct"].apply(bucket)
agg = g.groupby("surprise_bucket").agg(
    n=("ticker", "size"),
    **{f"avg_{h}d_%": (f"fwd_ret_{h}d", "mean") for h in HORIZONS},
    **{f"winrate_{h}d": (f"up_{h}d", "mean") for h in HORIZONS},
).round(3)
agg

## 6. 训练模型（方向分类 + 幅度回归）

按**时间顺序**划分训练/测试（用过去预测最近），避免未来数据泄露。

In [ ]:
results = {}
for h in HORIZONS:
    r = et.train_models(df, horizon=h, test_frac=0.25)
    results[h] = r
    print(f"===== {h}-day horizon =====")
    if r.clf is None:
        print(r.report); print()
        continue
    print(f"方向准确率  test={r.test_accuracy:.3f}  vs 基线(多数类)={r.baseline_accuracy:.3f}")
    print(f"幅度回归  R^2={r.reg_r2:.3f}  MAE={r.reg_mae:.2f}%")
    print("特征重要性:")
    print(r.importances.round(3).to_string())
    print()

In [ ]:
# 特征重要性可视化
valid = [h for h in HORIZONS if results[h].clf is not None]
if valid:
    fig, axes = plt.subplots(1, len(valid), figsize=(6 * len(valid), 4))
    if len(valid) == 1: axes = [axes]
    for ax, h in zip(axes, valid):
        imp = results[h].importances.sort_values()
        ax.barh(imp.index, imp.values, color="steelblue")
        ax.set_title(f"{h}-day: feature importance")
    plt.tight_layout(); plt.show()
else:
    print("样本不足，无法画特征重要性——多加几只股票到 TICKERS 重试。")

## 7. 对当前股票打分（预测下一次/最近财报后的走势）

用最新已公布财报的利润特征，给出上涨概率与预期收益。

In [ ]:
h = 5 if 5 in results and results[5].clf is not None else HORIZONS[0]
model = results[h]
scores = []
for t in TICKERS:
    s = et.score_upcoming(t, model)
    if s: scores.append(s)
import pandas as pd
board = pd.DataFrame(scores).sort_values("prob_up", ascending=False).reset_index(drop=True)
print(f"基于 {h}-天模型的打分（按上涨概率排序）:")
board

## 局限与下一步

- **样本量**：每只股票只有十几个季度财报，想要统计显著性请把 `TICKERS` 扩到几十、上百只。
- **财报时点**：盘前/盘后公布会影响入场日。本模型统一取“公告日当天或之后的第一个交易日”为入场，衡量随后 3/5 日漂移。
- **更好的验证**：可换成 walk-forward / 滞后交叉验证，并加入交易成本象限。
- **数据源**：若已授权 Robinhood 连接器，可把 `get_earnings_dates` / `get_quarterly_income` 换成 Robinhood 的 `get_earnings_results` / `get_financials`（见 README）。
- 本 notebook 仅用于研究，**不构成投资建议**。